# Reconstruction Visualization

Hamburg 1024×1024 tile visual comparison across architectures (ResSHyp / FP) and backends
(GPU FP32 / FPGA INT8) for selected λ values.

Data sources:
- **GPU** — FP32 reconstructions from W&B run directories (`run_dir`).
- **FPGA** — INT8 reconstructions from `compiled_models/<name>/results/<tile>_recon_linA.npy`.

All images shown as log-intensity, clipped per-image to mean ± 3σ.

In [ ]:
import json
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import torch

ROOT_DIR = Path("..").resolve()
sys.path.insert(0, str(ROOT_DIR))

from src.utils.metrics import psnr as _src_psnr, ssim as _src_ssim

# ── Paths ─────────────────────────────────────────────────────────────────────
WANDB_CSV = ROOT_DIR / "notebooks" / "SAR_DDC_FPGA_all_runs_WandB.csv"
COMPILED_MODELS_DIR = ROOT_DIR / "results" / "fpga" / "compiled_models"
PLOTS_DIR = ROOT_DIR / "results" / "plots" / "reconstruction"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
SAVE_FIGURES = True

# ── Shared conventions, palette, rendering ────────────────────────────────────
sys.path.insert(0, str(ROOT_DIR / "notebooks"))
from _plotkit import (
    PALETTE,
    ARCH_LABEL,
    BACKEND_COLORS,
    PROD_LR,
    linA_to_logI,
    show_logI,
    export_manuscript,
)

ARCH_LBL = ARCH_LABEL  # alias used throughout this notebook
ACCEPTED_SEEDS = [0, 1, 2, 3, 4, 5]

r, g, b, y, e = "\033[31m", "\033[32m", "\033[34m", "\033[33m", "\033[0m"

In [ ]:
# ── Load GPU runs (FP32) ──────────────────────────────────────────────────────
# Uses _plotkit.load_quality_runs; result includes run_dir for GPU recon .npy paths.
sys.path.insert(0, str(ROOT_DIR / "notebooks"))
from _plotkit import load_quality_runs, load_fpga_quality

gpu_df = load_quality_runs(WANDB_CSV, lr="prod", seeds=ACCEPTED_SEEDS, verbose=True)
fpga_df = load_fpga_quality(
    gpu_df=gpu_df, compiled_dir=COMPILED_MODELS_DIR, seeds=ACCEPTED_SEEDS, verbose=True
)

# ── Merge into one tidy df ────────────────────────────────────────────────────
df = pd.concat([gpu_df, fpga_df], ignore_index=True)
df["lambda"] = df["lambda"].astype(float)
df["seed"] = df["seed"].astype(int)
df = df.sort_values(["lambda", "seed", "backend"]).reset_index(drop=True)
print(f"\nCombined: {len(df)} rows ({df['backend'].value_counts().to_dict()})")

## Hamburg Tile Visualization

Visual comparison on the **1024 × 1024 Hamburg tile** across selected λ values.

**Layout** — one reference row + one FPGA/GPU row-pair per architecture in `ARCHS_FOR_VIS`:
- 1 arch → 3 rows (refs / FPGA / GPU)
- 2 archs → 5 rows (refs / FPGA arch₁ / GPU arch₁ / FPGA arch₂ / GPU arch₂)

All images are log-I, clipped per-image to mean ± 3σ.
PSNR is recomputed on-the-fly from the `.npy` arrays vs the MERLIN reference.

In [ ]:
# ── Data loading config ───────────────────────────────────────────────────────
TILE_NAME = "Hamburg_[11000:12024-8500:9524]"
SEED_FOR_VIS = 0  # seed used for all reconstructions (change + re-run to switch)
SHOW_ADAM_NOC = False  # load ADAM-NOC reference panel
SHOW_MERLIN_DDS = False  # load MERLIN-DDS reference panel
# ─────────────────────────────────────────────────────────────────────────────

REFS_DIR = ROOT_DIR / "data" / "visualization" / TILE_NAME
SUBPLOT_IN = 2.6  # default inches per subplot

# ── Aliases ───────────────────────────────────────────────────────────────────
_linA_to_logI = linA_to_logI
_show = show_logI


def _compute_psnr_tile(recon: np.ndarray, ref: np.ndarray) -> float:
    r_ = torch.from_numpy(recon.astype(np.float32))
    t_ = torch.from_numpy(ref.astype(np.float32))
    return _src_psnr(r_, t_)


def _compute_ssim_tile(recon: np.ndarray, ref: np.ndarray) -> float:
    r_ = torch.from_numpy(recon.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    t_ = torch.from_numpy(ref.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    return _src_ssim(r_, t_)


# ── Load reference panels ─────────────────────────────────────────────────────
noisy_linA = np.load(REFS_DIR / "linA_Noisy.npy")
merlin_linA = np.load(REFS_DIR / "linA_MERLIN.npy")
adam_noc_linA = np.load(REFS_DIR / "linA_ADAM_NOC.npy") if SHOW_ADAM_NOC else None
merlin_dds_linA = np.load(REFS_DIR / "linA_MERLIN_DDS.npy") if SHOW_MERLIN_DDS else None

ref_panels: List[Tuple[str, np.ndarray]] = [("Noisy", noisy_linA), ("MERLIN", merlin_linA)]
if adam_noc_linA is not None:
    ref_panels.append(("ADAM-NOC", adam_noc_linA))
if merlin_dds_linA is not None:
    ref_panels.append(("MERLIN-DDS", merlin_dds_linA))

# ── Load FPGA & GPU reconstructions for ALL archs and lambdas ────────────────
# Pre-load everything so plot_hamburg_grid() can be called with any arch/lambda subset.
all_archs = sorted(df["arch"].dropna().unique().tolist())
all_lambdas = sorted(df["lambda"].dropna().unique().tolist())

tile_data: Dict[str, Dict[float, Dict[str, Optional[Dict]]]] = {}

for _arch in all_archs:
    tile_data[_arch] = {}
    for lmbda in all_lambdas:
        tile_data[_arch][lmbda] = {}
        for backend in ["fpga", "gpu"]:
            rows = df[
                (df["lambda"] == lmbda)
                & (df["seed"] == SEED_FOR_VIS)
                & (df["backend"] == backend)
                & (df["arch"] == _arch)
            ]
            if len(rows) == 0:
                tile_data[_arch][lmbda][backend] = None
                continue
            row = rows.iloc[0]

            if backend == "gpu":
                if not row["run_dir"]:
                    raise ValueError(
                        f"GPU run for λ={int(lmbda)} seed={SEED_FOR_VIS} arch={_arch} has no "
                        "run_dir. Re-evaluate the run to generate Hamburg tile reconstructions."
                    )
                npy_path = Path(row["run_dir"]) / f"recon_{TILE_NAME}_linA.npy"
                metrics_path = Path(row["run_dir"]) / f"recon_{TILE_NAME}_metrics.json"
            else:
                npy_path = Path(row["model_dir"]) / "results" / f"{TILE_NAME}_recon_linA.npy"
                metrics_path = Path(row["model_dir"]) / "results" / f"{TILE_NAME}_metrics.json"

            if not npy_path.exists():
                print(
                    f"  {y}MISSING{e}: λ={int(lmbda)} seed={SEED_FOR_VIS} "
                    f"{backend} {_arch} — {npy_path.name}"
                )
                tile_data[_arch][lmbda][backend] = None
                continue

            linA = np.load(npy_path)
            psnr = _compute_psnr_tile(linA, merlin_linA)
            ssim = _compute_ssim_tile(linA, merlin_linA)
            bpp = float("nan")
            if metrics_path.exists():
                m = json.loads(metrics_path.read_text())
                bpp = float(m.get("bpp" if backend == "fpga" else "bpp_bitstream", float("nan")))

            tile_data[_arch][lmbda][backend] = {
                "linA": linA,
                "logI": _linA_to_logI(linA),
                "psnr_merlin": psnr,
                "ssim_merlin": ssim,
                "bpp": bpp,
            }

n_present = sum(
    1
    for arch_d in tile_data.values()
    for lam_d in arch_d.values()
    for v in lam_d.values()
    if v is not None
)
print(
    f"Loaded {n_present} tiles across {len(all_archs)} archs × {len(all_lambdas)} λ × 2 backends"
    f"  (seed={SEED_FOR_VIS})"
)

In [ ]:
def plot_hamburg_grid(
    tile_data: Dict,
    ref_panels: List[Tuple[str, np.ndarray]],
    *,
    archs: Optional[List[str]] = None,
    lambdas: Optional[List[float]] = None,
    seed: int = SEED_FOR_VIS,
    show_frames: bool = False,
    frame_color_by: str = "arch",
    closeup_roi: Optional[Tuple[int, int, int, int]] = None,
    subplot_in: float = SUBPLOT_IN,
    save: bool = True,
    manuscript_name: Optional[str] = None,
    no_title: bool = False,
) -> None:
    """Draw Hamburg tile reconstruction grid (and optionally a close-up).

    Produces one figure always, and a second cropped figure if *closeup_roi* is given.
    A red dashed rectangle is drawn on the Noisy reference panel to mark the close-up region.

    Args:
        tile_data:       Nested dict arch → lambda → backend → data (from data-loading cell).
        ref_panels:      List of (name, linA_array) reference panels.
        archs:           Architectures to include; None → all keys in tile_data.
        lambdas:         λ values to show; None → all available in tile_data.
        seed:            Included in the figure suptitle.
        show_frames:     Draw coloured borders around each reconstruction cell.
        frame_color_by:  "arch" (default) — border colour = PALETTE["architectures"][arch];
                         "backend" — border colour = BACKEND_COLORS[backend].
        closeup_roi:     (r0, r1, c0, c1) — if given, also produce a cropped close-up figure.
        subplot_in:      Inches per subplot cell.
        save:            Save PDFs to PLOTS_DIR.
        manuscript_name: If given, export the main (non-closeup) figure to MANUSCRIPT_DIR.
    """
    _archs = archs if archs is not None else list(tile_data.keys())
    if lambdas is None:
        _lambdas = sorted({lam for a in _archs for lam in tile_data.get(a, {}).keys()})
    else:
        _lambdas = sorted(lambdas)

    def _draw(crop_fn=None):
        n_refs = len(ref_panels)
        n_lam = len(_lambdas)
        n_cols = max(n_refs, n_lam)
        n_rows = 1 + 2 * len(_archs)

        row_headers = [(0, "References", "black")]
        for _i, _arch in enumerate(_archs):
            _arch_label = ARCH_LBL.get(_arch, _arch)
            row_headers += [
                (1 + 2 * _i, f"{_arch_label} - FPGA", "black"),
                (2 + 2 * _i, f"{_arch_label} - GPU", "black"),
            ]

        fig, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=(subplot_in * n_cols + 0.5, subplot_in * n_rows + 0.7),
            squeeze=False,
        )

        for _ri, _label, _color in row_headers:
            axes[_ri, 0].text(
                -0.12,
                0.5,
                _label,
                transform=axes[_ri, 0].transAxes,
                rotation=90,
                va="center",
                ha="center",
                fontsize=9,
                fontweight="bold",
                color=_color,
            )

        # Row 0: Reference panels
        for ci, (ref_name, ref_linA) in enumerate(ref_panels):
            _img = _linA_to_logI(ref_linA)
            if crop_fn:
                _img = crop_fn(_img)
            _show(axes[0, ci], _img, ref_name)
            if closeup_roi and not crop_fn and ref_name == "Noisy":
                r0, r1, c0, c1 = closeup_roi
                axes[0, ci].add_patch(
                    Rectangle(
                        (c0, r0),
                        c1 - c0,
                        r1 - r0,
                        linewidth=1.5,
                        edgecolor="red",
                        facecolor="none",
                        linestyle="--",
                    )
                )
        for ci in range(n_refs, n_cols):
            axes[0, ci].axis("off")

        # FPGA + GPU reconstruction rows
        for _i, _arch in enumerate(_archs):
            _arch_color = PALETTE["architectures"].get(_arch, "#999999")
            for _j, backend in enumerate(["fpga", "gpu"]):
                row_idx = 1 + 2 * _i + _j
                if frame_color_by == "arch":
                    color = _arch_color
                else:
                    color = BACKEND_COLORS[backend]
                for ci, lmbda in enumerate(_lambdas):
                    data = tile_data.get(_arch, {}).get(lmbda, {}).get(backend)
                    if data is None:
                        axes[row_idx, ci].axis("off")
                        axes[row_idx, ci].text(
                            0.5,
                            0.5,
                            "N/A",
                            ha="center",
                            va="center",
                            transform=axes[row_idx, ci].transAxes,
                            fontsize=10,
                            color="gray",
                        )
                        continue
                    bpp_str = f"{data['bpp']:.3f}" if not np.isnan(data["bpp"]) else "N/A"
                    ssim_str = (
                        f"{data['ssim_merlin']:.3f}"
                        if not np.isnan(data["ssim_merlin"])
                        else "N/A"
                    )
                    subtitle = f"BPP={bpp_str}  PSNR={data['psnr_merlin']:.2f}dB  SSIM={ssim_str}"
                    _img = data["logI"] if not crop_fn else crop_fn(data["logI"])
                    _show(
                        axes[row_idx, ci],
                        _img,
                        f"λ={int(lmbda)}",
                        subtitle,
                        border_color=(color if show_frames else None),
                        subtitle_fontsize=8,
                    )
                for ci in range(n_lam, n_cols):
                    axes[row_idx, ci].axis("off")

        arch_str = " + ".join(ARCH_LBL.get(a, a) for a in _archs)
        is_closeup = crop_fn is not None and closeup_roi is not None
        if is_closeup:
            r0, r1, c0, c1 = closeup_roi
            suptitle = (
                f"Hamburg Tile — Close-up  rows [{r0}:{r1}]  cols [{c0}:{c1}]\n"
                f"log-I · per-image mean±3σ  |  PSNR  |  seed={seed}  |  {arch_str}"
            )
            fname = f"Visualizations_Hamburg_closeup_{'_'.join(_archs)}.pdf"
            ms_name = None
        else:
            suptitle = (
                f"Hamburg Tile  —  {TILE_NAME}\n"
                f"log-I · per-image mean±3σ  |  PSNR  |  seed={seed}  |  {arch_str}"
            )
            fname = f"Visualizations_Hamburg_{'_'.join(_archs)}.pdf"
            ms_name = manuscript_name

        if not no_title:
            fig.suptitle(suptitle, fontsize=10, y=1.01)
        plt.tight_layout(h_pad=1.2, w_pad=0.3)

        if save:
            out = PLOTS_DIR / fname
            fig.savefig(out, bbox_inches="tight")
            print(f"Saved: {out}")
        if ms_name:
            export_manuscript(fig, ms_name)
        plt.show()

    _draw()
    if closeup_roi:
        r0, r1, c0, c1 = closeup_roi
        _draw(crop_fn=lambda arr: arr[r0:r1, c0:c1])

In [ ]:
# ── Paper figure: ResSH + FP, λ = 1/20/1000, with close-up ──────────────────
plot_hamburg_grid(
    tile_data,
    ref_panels,
    archs=["ResSHyp", "FP"],
    lambdas=[1, 20, 1000],
    seed=SEED_FOR_VIS,
    show_frames=True,
    # closeup_roi=(0, 200, 100, 300),
    save=SAVE_FIGURES,
    manuscript_name="fig_qualitative_grid",
    no_title=True,
)

## Additional examples

Call `plot_hamburg_grid` with different arguments to explore other combinations without re-running the data loader.

In [ ]:
# ── Examples: swap archs, lambdas, or disable close-up ───────────────────────

# All 4 archs, 3 representative λ, no close-up:
# plot_hamburg_grid(tile_data, ref_panels, lambdas=[1, 20, 1000])

# Single arch, all lambdas (diagnostic sweep):
# plot_hamburg_grid(tile_data, ref_panels, archs=["ResSHyp"], lambdas=None)

# Arch-coloured borders (default), different crop region:
plot_hamburg_grid(
    tile_data,
    ref_panels,
    archs=["ResSHyp", "SHyp"],
    lambdas=[1, 20, 1000],
    show_frames=True,
    frame_color_by="arch",  # or "backend" to colour by FPGA/GPU
    closeup_roi=(300, 500, 200, 400),
)